# Tutorial: Off-Policy Evaluation with `pcmabinf`

This notebook walks through a complete off-policy evaluation (OPE) workflow using the `pcmabinf` package. By the end you will be able to:

1. Frame your problem as a contextual bandit
2. Collect logged bandit data with a logging policy
3. Define one or more target policies to evaluate
4. Run all OPE estimators (DM, IPS, DR, ADR, CADR, MRDR, CAMRDR)
5. Measure and interpret 95% CI coverage across simulation replicates
6. Visualise the results

---

## When should you use this package?

You have a **contextual bandit problem**: at each step your system observes some context (user features, patient covariates, request attributes, …), selects an action (recommendation, treatment, ad, …), and observes a reward (click, outcome, revenue, …).

You have deployed one policy (the **logging policy**) and collected a dataset. Now you want to know: *how well would a different policy (the **target policy**) have performed on that same data?* That is OPE.

The challenge is that you only observed the reward for the action the logging policy chose, not for all possible actions. OPE estimators correct for this selection bias using the known action probabilities of the logging policy.

---

## Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

from pcmabinf import (
    BanditData,
    LoggingConfig,
    OPEEstimator,
    OpenMLCC18World,
    compute_coverage_metrics,
    make_target_policy,
    run_bandit_simulations,
    run_ope_simulations_multipolicy,
)
from pcmabinf.viz import bar_plot

DATA_DIR = Path.home() / "pcmabinf" / "OpenML-CC18"
TASK_ID  = "11"   # iris: 150 samples, 4 features, 3 arms — fast for a tutorial
SEED     = 42

---

## 1. The bandit world

Each OpenML-CC18 task is a classification dataset repurposed as a contextual bandit:

- **Context** `x` — the feature vector for one observation
- **Arms** — the class labels (0, 1, …, K-1)
- **Reward** — 1 if the chosen arm matches the true label, 0 otherwise, plus optional Gaussian noise

If you have your own dataset you can adapt this pattern: wrap it in a class that exposes `arm_count`, `sample_contexts()`, `rewards_batch()`, and `regrets_batch()`.

`reward_variance=1.0` adds Gaussian noise to rewards, matching the paper's setup.

In [ ]:
world = OpenMLCC18World(
    task_id=TASK_ID,
    data_dir=DATA_DIR,
    reward_variance=1.0,
)

print(f"Task {TASK_ID}")
print(f"  Arms (classes) : {world.arm_count}")
print(f"  Features       : {world.feature_count}")
print(f"  Observations   : {world.observation_count}")

---

## 2. Collecting data with the logging policy

The logging policy is the policy that was (or will be) deployed to collect data. It must be **stochastic** — it must choose each arm with non-zero probability — so that the importance weights used by IPS/DR estimators are well-defined.

We use an **epsilon-greedy** policy:
- In the first batch (no data yet) it explores uniformly at random.
- From batch 2 onward it fits a per-arm outcome model, picks the greedy arm with probability 1 − ε, and explores uniformly with probability ε.
- The exploration rate decays as ε_t = `epsilon_multiplier` / (n_t + 1)^(1/3).

`propensity_history` stores, after each batch, the re-evaluated action probabilities for all observations collected so far under the current model. The CADR and CAMRDR estimators use this to correct for distributional shift in the logging policy over time.

In [ ]:
config = LoggingConfig(
    batch_count=20,       # number of sequential batches
    batch_size=50,        # observations per batch
    strategy="greedy",    # epsilon-greedy; use "uniform" for pure exploration
    epsilon_multiplier=0.01,
)

# Run 16 independent replicates in parallel.
# Each replicate produces one BanditData object.
bandit_data_list: list[BanditData] = run_bandit_simulations(
    world, config, n_simulations=16, n_jobs=-1, seed=SEED
)

d = bandit_data_list[0]  # inspect the first replicate
N = len(d.A)
print(f"Collected {N} observations over {config.batch_count} batches")
print(f"X shape : {d.X.shape}")
print(f"Arms chosen  : {np.unique(d.A, return_counts=True)}")
print(f"Mean reward  : {d.Y.mean():.3f}")
print(f"Mean regret  : {d.regret.mean():.3f}  (0 = always optimal)")
print(f"Propensity range: [{d.P.min():.3f}, {d.P.max():.3f}]")

### Cumulative regret over time

A lower cumulative regret means the logging policy learned quickly which arm to pick. The shape of this curve tells you how much exploration vs exploitation happened.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
for bd in bandit_data_list:
    ax.plot(np.cumsum(bd.regret), color="steelblue", alpha=0.3, linewidth=0.8)
ax.set_xlabel("Step")
ax.set_ylabel("Cumulative regret")
ax.set_title(f"Task {TASK_ID} — cumulative regret across {len(bandit_data_list)} replicates")
plt.tight_layout()
plt.show()

---

## 3. Defining target policies

A target policy answers: *given context x, what is the probability of choosing each arm?*  It is represented by any object with a `pi(X) -> (N, K)` method.

Here we define four target policies to evaluate simultaneously:

| Name | Type | Description |
|---|---|---|
| `linear` | Contextual | Greedy policy using a linear outcome model |
| `tree` | Contextual | Greedy policy using a decision tree |
| `arm0` | Non-contextual | Always pick arm 0 |
| `arm1` | Non-contextual | Always pick arm 1 |

Contextual policies are trained on a fresh sample from the world (independent of the logged data) so they represent what a learner with access to the environment would do.

In [ ]:
# Use the same sample size as the logged data so the policy sees a
# comparable amount of training data.
train_size = len(bandit_data_list[0].X)

target_policies = {
    "linear": make_target_policy(
        "contextual", world,
        outcome_model=LinearRegression(),
        train_sample_size=train_size,
    ),
    "tree": make_target_policy(
        "contextual", world,
        outcome_model=DecisionTreeRegressor(),
        train_sample_size=train_size,
    ),
    "arm0": make_target_policy("non_contextual_constant_0", world),
    "arm1": make_target_policy("non_contextual_constant_1", world),
}

# Quick sanity check: pi() must return (N, K) probability matrices.
X_sample = world.sample_contexts(5)
for name, policy in target_policies.items():
    pi = policy.pi(X_sample)
    assert pi.shape == (5, world.arm_count)
    assert np.allclose(pi.sum(axis=1), 1.0)
    print(f"{name}: pi shape = {pi.shape}, row sums = {pi.sum(axis=1)}")

---

## 4. Running OPE for a single replicate

Before running all replicates in parallel, it is useful to inspect what `OPEEstimator` returns for one run. Each estimator returns a `(mean, variance)` tuple where:

- `mean` — the estimated policy value (expected reward under the target policy)
- `variance` — the variance of the estimator's mean (used to construct CIs)

The `truth` field is the oracle value computed directly from the world (not from the logged data), and has variance 0 by definition.

In [ ]:
from pcmabinf import OPEResult

single_data = bandit_data_list[0]
policy = target_policies["linear"]

result: OPEResult = OPEEstimator(
    data=single_data,
    world=world,
    target_policy=policy,
    outcome_model=LinearRegression(),
    n_folds=4,
).compute_all()

print("Single-replicate estimates for the linear target policy:")
print(f"  Truth  : {result.truth[0]:.4f}  (variance {result.truth[1]:.4f})")
print(f"  DM     : {result.dm[0]:.4f}  (variance {result.dm[1]:.4f})")
print(f"  IPS    : {result.ips[0]:.4f}  (variance {result.ips[1]:.4f})")
print(f"  DR     : {result.dr[0]:.4f}  (variance {result.dr[1]:.4f})")
print(f"  ADR    : {result.adr[0]:.4f}  (variance {result.adr[1]:.4f})")
print(f"  CADR   : {result.cadr[0]:.4f}  (variance {result.cadr[1]:.4f})")
print(f"  MRDR   : {result.mrdr[0]:.4f}  (variance {result.mrdr[1]:.4f})")
print(f"  CAMRDR : {result.camrdr[0]:.4f}  (variance {result.camrdr[1]:.4f})")

---

## 5. Running OPE across all replicates and policies

`run_ope_simulations_multipolicy` evaluates all policies against all replicates in a single parallel pass. This is significantly faster than running one policy at a time because it avoids repeated worker pool initialisation.

The result is a dict mapping policy name → list of `OPEResult`, one per replicate.

In [ ]:
all_ope: dict[str, list[OPEResult]] = run_ope_simulations_multipolicy(
    bandit_data_list,
    world,
    target_policies,
    outcome_model=LinearRegression(),
    n_jobs=-1,
)

for policy_name, results in all_ope.items():
    print(f"{policy_name}: {len(results)} replicates")

---

## 6. Computing and interpreting coverage metrics

For each replicate *s* and estimator *e*, we form a 95% confidence interval:

```
CI_s = [mean_e - 1.96 * sqrt(var_e),  mean_e + 1.96 * sqrt(var_e)]
```

**Coverage** is the fraction of replicates where `CI_s` contains the oracle truth value.

- Coverage ≈ 0.95 → well-calibrated (CIs are correctly sized)
- Coverage < 0.95 → under-coverage (CIs are too narrow; the estimator is overconfident)
- Coverage > 0.95 → over-coverage (CIs are too wide; the estimator is conservative)

CADR and CAMRDR are designed to achieve better coverage than DR/IPS, especially when the logging policy changes over time.

In [ ]:
ESTIMATORS = ["dm", "ips", "dr", "adr", "cadr", "mrdr", "camrdr"]

print(f"{'Policy':<10}  {'Estimator':<10}  {'Coverage':>10}  {'Std err':>10}")
print("-" * 46)
for policy_name, ope_results in all_ope.items():
    metrics = compute_coverage_metrics(ope_results)
    for est in ESTIMATORS:
        cov  = float(metrics[f"{est}_mean"])
        serr = float(metrics[f"{est}_stderr"])
        flag = "  ✓" if abs(cov - 0.95) < 0.05 else ""
        print(f"{policy_name:<10}  {est.upper():<10}  {cov:>10.3f}  {serr:>10.3f}{flag}")
    print()

---

## 7. Visualising the results

### Bar chart of coverage

`bar_plot` takes an array of shape `(n_estimators, n_replicates)` and plots means with standard-error bars. Here we plot coverage per estimator for the linear policy.

In [ ]:
import numpy as np
from scipy.stats import norm

policy_name = "linear"
ope_results = all_ope[policy_name]
z = norm.ppf(0.975)

# Build (n_estimators, n_replicates) coverage matrix.
# For each replicate, a CI either covers the truth (1) or not (0).
truth_means = np.array([r.truth[0] for r in ope_results])

coverage_matrix = []
for est in ESTIMATORS:
    hits = []
    for r, truth in zip(ope_results, truth_means):
        val = getattr(r, est)
        if val is not None:
            lo = val[0] - z * np.sqrt(max(val[1], 0))
            hi = val[0] + z * np.sqrt(max(val[1], 0))
            hits.append(float(lo <= truth <= hi))
    coverage_matrix.append(hits)

data = np.array(coverage_matrix)  # (n_estimators, n_replicates)

ax = bar_plot(data, metric="95% CI Coverage", labels=ESTIMATORS, truth=0.95)
ax.set_title(f"Task {TASK_ID} — {policy_name} target policy")
plt.tight_layout()
plt.show()

### Policy value estimates

We can also plot the estimated policy *value* (expected reward) rather than coverage, to see how close each estimator is to the truth.

In [ ]:
# Build (n_estimators, n_replicates) matrix of policy value estimates.
value_matrix = []
for est in ESTIMATORS:
    values = [getattr(r, est)[0] for r in ope_results if getattr(r, est) is not None]
    value_matrix.append(values)

value_data = np.array(value_matrix)
truth_mean = float(np.mean(truth_means))

ax = bar_plot(value_data, metric="Estimated policy value", labels=ESTIMATORS, truth=truth_mean)
ax.set_title(f"Task {TASK_ID} — {policy_name} target policy")
plt.tight_layout()
plt.show()

---

## 8. Scaling up: running across multiple tasks

To reproduce the paper's full benchmark, run the CLI:

```bash
make reproduce
```

Or from Python, loop over tasks and accumulate results:

```python
import pandas as pd

task_ids = ["11", "14", "15", "16"]  # or read from DATA_DIR
rows = []

for task_id in task_ids:
    world = OpenMLCC18World(task_id=task_id, data_dir=DATA_DIR, reward_variance=1.0)
    bandit_data_list = run_bandit_simulations(world, config, n_simulations=64, n_jobs=-1)
    train_size = len(bandit_data_list[0].X)
    target_policies = {
        "linear": make_target_policy("contextual", world,
                                     outcome_model=LinearRegression(),
                                     train_sample_size=train_size),
        "arm0": make_target_policy("non_contextual_constant_0", world),
    }
    all_ope = run_ope_simulations_multipolicy(
        bandit_data_list, world, target_policies,
        outcome_model=LinearRegression(), n_jobs=-1,
    )
    row = {"task_id": task_id}
    for name, ope_results in all_ope.items():
        metrics = compute_coverage_metrics(ope_results)
        for key, val in metrics.items():
            row[f"{name}_{key}"] = float(val)
    rows.append(row)

df = pd.DataFrame(rows)
df.to_pickle("results/my_results")
```

Then visualise with:

```bash
uv run pcmabinf visualize results/my_results
```

---

## 9. Bringing your own data

If you have your own logged bandit dataset rather than an OpenML task, construct a `BanditData` object directly and pass it to `OPEEstimator`:

```python
import numpy as np
from pcmabinf import BanditData, OPEEstimator

# Your logged data
N, d, K = 1000, 10, 3
data = BanditData(
    X=np.random.randn(N, d),           # contexts
    A=np.random.randint(0, K, N),      # chosen arms
    Y=np.random.randn(N),              # observed rewards
    P=np.full(N, 1.0 / K),             # logging propensities P(A|X)
    propensity_history=[               # for CADR: one entry per batch
        np.full(N, 1.0 / K)
    ],
    epsilon=np.full(N, 1.0),
    regret=np.zeros(N, dtype=np.intp),
    batch_size=N,
)

# Define your target policy
class MyPolicy:
    def pi(self, X):
        # Return (N, K) probability matrix — must sum to 1 per row
        probs = np.zeros((len(X), K))
        probs[:, 0] = 1.0   # always pick arm 0
        return probs

# Run OPE — pass None for world if you don't need the truth estimator
# (truth requires access to the ground-truth reward function)
estimator = OPEEstimator(
    data=data,
    world=None,          # set to None if you don't have a world object
    target_policy=MyPolicy(),
    outcome_model=LinearRegression(),
)
result = estimator.compute_all()
print(f"CADR estimate: {result.cadr[0]:.4f} ± {np.sqrt(result.cadr[1]):.4f}")
```

> **Note on `propensity_history`**: if your logging policy did not change over time (static policy), pass a list with a single entry: the propensity vector for all observations. If your policy was updated in batches (as in the epsilon-greedy setup above), each entry should be the re-evaluated propensities for all observations collected so far under the model from that batch.

---

## Summary

| Step | Function / Class |
|---|---|
| Define environment | `OpenMLCC18World` or custom |
| Configure logging policy | `LoggingConfig` |
| Collect data | `run_bandit_simulations` |
| Define target policy | `make_target_policy` or custom class with `.pi()` |
| Run OPE | `run_ope_simulations_multipolicy` |
| Measure coverage | `compute_coverage_metrics` |
| Visualise | `bar_plot`, `visualize_coverage` |